In [4]:
# multivariate_ehrhart_fd_fixed.sage
# Multivariate Ehrhart via finite differences (Sage) -- corrected conversion

from sage.all import *
from itertools import product

# ------------------ helpers -------------------------------------------------

def poly_from_vertices(verts):
    return Polyhedron(vertices=[vector(ZZ, v) for v in verts])

def minkowski_sum_scaled(polys, ns):
    S = None
    for P, n in zip(polys, ns):
        if n == 0:
            continue
        Q = P.dilation(n)
        S = Q if S is None else S.minkowski_sum(Q)
    if S is None:
        ambient = polys[0].ambient_dim()
        S = Polyhedron(vertices=[(0,)*ambient])
    return S

def lattice_count(poly):
    try:
        return Integer(poly.integral_points_count())
    except Exception:
        return Integer(len(poly.integral_points()))

def multiindex_cube(k, D):
    return list(product(range(D+1), repeat=k))

def multiindices_leq_degree(k, D):
    res = []
    def rec(i, rem, curr):
        if i == k:
            res.append(tuple(curr))
        else:
            for a in range(rem + 1):
                curr.append(a)
                rec(i + 1, rem - a, curr)
                curr.pop()
    for tot in range(D + 1):
        rec(0, tot, [])
    return res

def forward_diff_coeffs(Evals_cube, alpha):
    """
    c_alpha = (Delta^alpha E)(0) with forward differences.
    """
    total = Integer(0)
    for j in product(*[range(a+1) for a in alpha]):
        sign = (-1)**(sum(alpha) - sum(j))
        mult = Integer(1)
        for ai, ji in zip(alpha, j):
            mult *= Integer(binomial(ai, ji))
        total += Integer(sign) * mult * Integer(Evals_cube[j])
    return total

def stirling1_table(nmax):
    """
    Signed Stirling numbers of the first kind s(n,k).
    s(0,0)=1, recurrence: s(n,k)=s(n-1,k-1) - (n-1)*s(n-1,k)
    """
    S1 = [[Integer(0)]*(nmax+1) for _ in range(nmax+1)]
    S1[0][0] = Integer(1)
    for n in range(1, nmax+1):
        for k in range(1, n+1):
            S1[n][k] = S1[n-1][k-1] - Integer(n-1)*S1[n-1][k]
    return S1

def binomial_to_monomial(coeff_bin, k, D):
    """
    Correct conversion from binomial basis to monomial basis using
    signed Stirling numbers of the first kind.
    coeff_bin: dict alpha -> c_alpha such that
      E(n) = sum_alpha c_alpha * prod_i binom(n_i, alpha_i).
    Returns coeff_mono: dict alpha -> rational coefficient of prod_i n_i^{alpha_i}.
    Formula (multivariate):
      coeff_mono[alpha] = sum_{beta >= alpha} c_beta * prod_i ( s(beta_i, alpha_i) / beta_i! )
    """
    S1 = stirling1_table(D)
    coeff_mono = {}
    A = multiindices_leq_degree(k, D)
    for alpha in A:
        ssum = QQ(0)
        for beta, c in coeff_bin.items():
            ok = True
            for i in range(k):
                if beta[i] < alpha[i]:
                    ok = False; break
            if not ok:
                continue
            prod_factor = QQ(1)
            for i in range(k):
                prod_factor *= QQ(S1[beta[i]][alpha[i]]) / QQ(factorial(beta[i]))
            ssum += QQ(c) * prod_factor
        coeff_mono[alpha] = ssum
    return coeff_mono

# ------------------ main function ------------------------------------------

def multivariate_ehrhart(polys):
    k = len(polys)
    d = polys[0].ambient_dim()
    D = d
    EvalPts = multiindex_cube(k, D)
    Evals = {}
    for beta in EvalPts:
        Psum = minkowski_sum_scaled(polys, beta)
        Evals[beta] = lattice_count(Psum)
    A = multiindices_leq_degree(k, D)
    coeff_bin = {}
    for alpha in A:
        coeff_bin[alpha] = forward_diff_coeffs(Evals, alpha)
    coeff_mono = binomial_to_monomial(coeff_bin, k, D)
    R = PolynomialRing(QQ, ['n{}'.format(i+1) for i in range(k)])
    ns = R.gens()
    Epoly = R(0)
    for alpha, c in coeff_bin.items():
        term = R(c)
        for i, a in enumerate(alpha):
            term *= ns[i]**a
        Epoly += term
    return Epoly, coeff_bin


In [29]:
",".join(['n{}'.format(i+1) for i in range(6)])

'n1,n2,n3,n4,n5,n6'

In [26]:
R = PolynomialRing(QQ, ['x1', 'x2'])
f = 2*x1^2 + x1*x2 + 3*x2 + 4
f.hessian()

[4 1]
[1 0]

In [5]:
from collections import defaultdict
if __name__ == "__main__":
    P1 = poly_from_vertices([(0, 0, 0), (1, 0, 0)])
    P2 = poly_from_vertices([(0, 0, 0), (0, 1, 0)])
    P3 = poly_from_vertices([(0, 0, 0), (0, 0, 1)])
    P4 = poly_from_vertices([(0, 0, 0), (0, 1, 0), (1, 0, 0), (1,1,0)])
    P5 = poly_from_vertices([(0, 0, 0), (0, 0, 1), (1, 0, 0),(1,0,1)])
    P6 = poly_from_vertices([(0, 0, 0), (0, 1, 0), (0, 0, 1),(0,1,1)])
    #P2 = poly_from_vertices([(0, 0, 0), (2, 0, 0), (2, 1, 0), (0, 1, 0), (0, 0, 4), (2, 1, 4), (0, 1, 4), (2, 0, 4)])
    #P3 = poly_from_vertices([(0, 0, 0), (4, 0, 0), (4, 3, 0), (0, 3, 0), (0, 0, 1), (4, 3, 1), (0, 3, 1), (4, 0, 1)])
    E, coeff_bin = multivariate_ehrhart([P1, P2, P3, P4, P5, P6])
    def dPart( F, d ):
        return sum( [ F.monomial_coefficient(m) * m
                   for m in F.monomials() 
                   if  m.total_degree() == d ] )
        
    d = defaultdict(E.parent())
    print("E =", E)
    deg2 = dPart(E, 2)
    print("deg 2 part:", deg2)
    #print("Hessian signature:", deg2.hessian().eigenvalues())
    print("Binomial-basis coefficients (alpha -> c_alpha):")
    for a in sorted(coeff_bin.keys()):
        print(a, "->", coeff_bin[a])
    #print("Monomial-basis coefficients (alpha -> coeff):")
    #for a in sorted(coeff_mono.keys()):
    #    print(a, "->", coeff_mono[a])
    print("Sanity checks:")
    print("E(0,0) =", E(0,0,0,0,0,0))
    print("E(1,0) should equal #P1 lattice pts (4):", E(1,0,0,0,0,0), lattice_count(P1))
    print("E(0,1) should equal #P2 lattice pts (3):", E(0,1,0,0,0,0), lattice_count(P2))
    print("E(1,1) should equal #(P1+P2):", E(1,1,0,0,0,0), lattice_count(minkowski_sum_scaled([P1,P2], (1,1))))

E = n1*n2*n3 + n1*n3*n4 + n2*n3*n4 + 2*n3*n4^2 + n1*n2*n5 + n2*n3*n5 + n1*n4*n5 + n2*n4*n5 + n3*n4*n5 + 2*n4^2*n5 + 2*n2*n5^2 + 2*n4*n5^2 + n1*n2*n6 + n1*n3*n6 + n1*n4*n6 + n2*n4*n6 + n3*n4*n6 + 2*n4^2*n6 + n1*n5*n6 + n2*n5*n6 + n3*n5*n6 + 2*n4*n5*n6 + 2*n5^2*n6 + 2*n1*n6^2 + 2*n4*n6^2 + 2*n5*n6^2 + n1*n2 + n1*n3 + n2*n3 + n1*n4 + n2*n4 + 3*n3*n4 + 2*n4^2 + n1*n5 + 3*n2*n5 + n3*n5 + 5*n4*n5 + 2*n5^2 + 3*n1*n6 + n2*n6 + n3*n6 + 5*n4*n6 + 5*n5*n6 + 2*n6^2 + n1 + n2 + n3 + 3*n4 + 3*n5 + 3*n6 + 1
deg 2 part: n1*n2 + n1*n3 + n2*n3 + n1*n4 + n2*n4 + 3*n3*n4 + 2*n4^2 + n1*n5 + 3*n2*n5 + n3*n5 + 5*n4*n5 + 2*n5^2 + 3*n1*n6 + n2*n6 + n3*n6 + 5*n4*n6 + 5*n5*n6 + 2*n6^2
Binomial-basis coefficients (alpha -> c_alpha):
(0, 0, 0, 0, 0, 0) -> 1
(0, 0, 0, 0, 0, 1) -> 3
(0, 0, 0, 0, 0, 2) -> 2
(0, 0, 0, 0, 0, 3) -> 0
(0, 0, 0, 0, 1, 0) -> 3
(0, 0, 0, 0, 1, 1) -> 5
(0, 0, 0, 0, 1, 2) -> 2
(0, 0, 0, 0, 2, 0) -> 2
(0, 0, 0, 0, 2, 1) -> 2
(0, 0, 0, 0, 3, 0) -> 0
(0, 0, 0, 1, 0, 0) -> 3
(0, 0, 0, 1, 0, 1) ->

In [14]:
var('n1,n2,n3,n4,n5,n6')
deg2 = n1*n2 + n1*n3 + n2*n3 + n1*n4 + n2*n4 + 3*n3*n4 + n4^2 + n1*n5 + 3*n2*n5 + n3*n5 + 5*n4*n5 + 4*n5^2 + 3*n1*n6 + n2*n6 + n3*n6 + 5*n4*n6 + 5*n5*n6 + 4*n6^2
[float(x) for x in deg2.hessian().eigenvalues()]

[0.5660188679433968,
 19.4339811320566,
 -1.8284271247461903,
 -1.8284271247461903,
 3.8284271247461903,
 3.8284271247461903]

In [9]:
deg2.hessian())

TypeError: float() argument must be a string or a real number, not 'sage.matrix.matrix_symbolic_dense.Matrix_symbolic_dense'

In [10]:
var('x1,x2,x3,x4,x5,x6')
f = x6^2 + 3*x5*x6 + x5^2 + 3*x4*x6 + 3*x4*x5 + x4^2 + x3*x6 + x3*x5 + 2*x3*x4 + x2*x6 + 2*x2*x5 + x2*x4 + x2*x3 + 2*x1*x6 + x1*x5 + x1*x4 + x1*x3 + x1*x2 

In [13]:
f.hessian()

[0 1 1 1 1 2]
[1 0 1 1 2 1]
[1 1 0 2 1 1]
[1 1 2 2 3 3]
[1 2 1 3 2 3]
[2 1 1 3 3 2]

In [12]:
f.hessian().eigenvalues()

[10, -2, -2, 0, 0, 0]

In [6]:
def random_lattice_simplex_vertices(dim, coord_bound):
    """Generate d-dimensional simplex as list of (dim+1) vertices (lists of ints)."""
    verts = []
    while len(verts) < dim + 1:
        v = [randint(-coord_bound, coord_bound) for _ in range(dim)]
        if v not in verts:
            verts.append(v)
    return verts

def multivariate_ehrhart_from_vertices(list_of_vertex_lists):
    """Call the original multivariate_ehrhart using vertex lists."""
    polys = [poly_from_vertices(verts) for verts in list_of_vertex_lists]
    return multivariate_ehrhart(polys)

def search_negative_mixed_terms_print(dim, num_polys, num_trials, coord_bound):
    """
    Fast search for negative mixed coefficients in multivariate Ehrhart polynomials.
    Prints detailed info when a negative mixed term is found.
    """
    negatives = []
    for t in range(num_trials):
        # Generate polytopes as vertex lists only
        vertex_lists = [random_lattice_simplex_vertices(dim, coord_bound) for _ in range(num_polys)]
        try:
            Epoly, coeff_mono, coeff_bin = multivariate_ehrhart_from_vertices(vertex_lists)
        except Exception:
            continue  # skip degenerate geometry
        print("bin:")
        for a in sorted(coeff_bin.keys()):
            print(a, "->", coeff_bin[a])
        print("mon:")
        for a in sorted(coeff_mono.keys()):
            print(a, "->", coeff_mono[a])
        # Check mixed terms (support >= 2)
        #for alpha, coeff in coeff_mono.items():
        #    if sum(alpha) > 1 and sum(1 for a in alpha if a > 0) > 1 and coeff < 0:
        #        negatives.append((Epoly, alpha, coeff, vertex_lists))
        #        for a in sorted(coeff_bin.keys()):
        #    
        #        # Print detailed info
        #        print(f"Trial {t}: Negative mixed coefficient {coeff} for term {alpha}")
        #        for i, verts in enumerate(vertex_lists):
        #            print(f"  Polytope {i+1} vertices: {verts}")
        #        break  # stop after first negative per trial
    #return negatives

In [1]:
# Search in dimension 4, 2 polytopes, 20 trials, coordinates in [-3,3]
#search_negative_mixed_terms_print(dim=5, num_polys=2, num_trials=1, coord_bound=2)
#print("Total negative mixed coefficient cases found:", len(negatives))

In [9]:
# SageMath ≥ 9.x

from sage.all import Polyhedron, ZZ
from itertools import product

# ----------------------------
# Safety & counting backends
# ----------------------------

def is_bounded_poly(P):
    try:
        return P.is_compact()
    except AttributeError:
        return (len(P.rays_list()) == 0 and len(P.lines_list()) == 0)

def integral_points_count_strict(P, allow_enumeration_highdim=False, fallback_list_threshold=200000):
    """
    Count lattice points in P with high-dim safety.
    - First try P.integral_points_count() (Barvinok/Normaliz backends).
    - If that fails:
        * In dim <= 3, fall back to enumerating.
        * In dim >= 4, only enumerate if allow_enumeration_highdim=True.
    """
    if P.is_empty():
        return 0
    d = P.ambient_dim()
    # Preferred fast path (uses installed backends)
    try:
        return ZZ(P.integral_points_count())
    except Exception:
        pass

    # Careful fallback
    if d >= 4 and not allow_enumeration_highdim:
        raise RuntimeError(
            "No fast integer-point backend available; "
            "refusing to enumerate in dim >= 4. "
            "Install latte-int/barvinok/normaliz or set allow_enumeration_highdim=True at your own risk."
        )

    pts = P.integral_points()
    if len(pts) > fallback_list_threshold:
        raise RuntimeError("Too many integer points to enumerate safely.")
    return ZZ(len(pts))

# ----------------------------
# Bivariate multiplicity grid precompute
# ----------------------------

def precompute_counts_bivariate(P, Q, max_a=3, max_b=3,
                                allow_enumeration_highdim=False):
    """
    Precompute counts[(a,b)] = #((aP + bQ) ∩ Z^d) for 0 ≤ a ≤ max_a, 0 ≤ b ≤ max_b.
    Uses caches for scaled pieces aP, bQ and for their Minkowski sums.
    """
    # Sanity checks
    if P.ambient_dim() != Q.ambient_dim():
        raise ValueError("P and Q must live in the same ambient dimension.")
    for R in (P, Q):
        if not is_bounded_poly(R):
            raise ValueError("Inputs must be bounded (polytopes).")

    d = P.ambient_dim()
    counts = {}
    scaled_cache = {}  # (('P',a) or ('Q',b)) -> scaled polytope
    sum_cache = {}     # (a,b) -> aP + bQ

    # Precache scaled polytopes we will need
    for a in range(max_a+1):
        if a == 0:
            scaled_cache[('P', 0)] = None  # treated specially
        else:
            scaled_cache[('P', a)] = P.dilation(a)

    for b in range(max_b+1):
        if b == 0:
            scaled_cache[('Q', 0)] = None
        else:
            scaled_cache[('Q', b)] = Q.dilation(b)

    # Build Minkowski sums and count
    for a in range(max_a+1):
        for b in range(max_b+1):
            key = (a, b)
            if a == 0 and b == 0:
                counts[key] = ZZ(1)  # {0}
                continue
            if key in sum_cache:
                S = sum_cache[key]
            else:
                AP = scaled_cache[('P', a)]
                BQ = scaled_cache[('Q', b)]
                if a == 0:
                    S = BQ
                elif b == 0:
                    S = AP
                else:
                    S = AP.minkowski_sum(BQ)
                sum_cache[key] = S

            if not is_bounded_poly(S):
                raise ValueError("Minkowski sum became unbounded; check inputs.")
            counts[key] = integral_points_count_strict(
                S,
                allow_enumeration_highdim=allow_enumeration_highdim
            )
    return counts

# ----------------------------
# CMV from multiplicities (bivariate)
# ----------------------------

def cmv_bivariate(counts, mP, mQ):
    """
    CMV for a multiset with mP copies of P and mQ copies of Q via inclusion–exclusion:
      CMV(mP,mQ) = sum_{0 ≤ a ≤ mP} sum_{0 ≤ b ≤ mQ}
                     (-1)^{k-(a+b)} * binom(mP,a) * binom(mQ,b) * counts[(a,b)],
      where k = mP + mQ, and counts[(0,0)] = 1 by convention.
    """
    k = mP + mQ
    total = ZZ(0)
    for a in range(mP + 1):
        for b in range(mQ + 1):
            coeff = binomial(mP, a) * binomial(mQ, b)
            sign  = -1 if ((k - (a + b)) % 2) else 1
            total += ZZ(sign) * ZZ(coeff) * ZZ(counts[(a, b)])
    return total

# ----------------------------
# Degree-3 slice checks
# ----------------------------

def degree3_slice_values(P, Q, allow_enumeration_highdim=False):
    """
    Compute the four needed CMVs:
      A = CMV(P[3], Q[0])
      B = CMV(P[2], Q[1])
      C = CMV(P[1], Q[2])
      D = CMV(P[0], Q[3])
    Returns (A,B,C,D).
    """
    counts = precompute_counts_bivariate(
        P, Q, max_a=3, max_b=3,
        allow_enumeration_highdim=allow_enumeration_highdim
    )
    A = cmv_bivariate(counts, 3, 0)
    B = cmv_bivariate(counts, 2, 1)
    C = cmv_bivariate(counts, 1, 2)
    D = cmv_bivariate(counts, 0, 3)
    return A, B, C, D

def check_degree3_inequalities(P, Q, allow_enumeration_highdim=True, verbose=True):
    """
    Checks:
      (1) CMV(P[3]) * CMV(P, Q[2]) <= CMV(P[2], Q)^2
      (2) CMV(P[2], Q) * CMV(Q[3]) <= CMV(P, Q[2])^2
    Returns dict with values and booleans, and optionally prints a summary.
    """
    A, B, C, D = degree3_slice_values(P, Q,
                                      allow_enumeration_highdim=allow_enumeration_highdim)
    ineq1_lhs = A * C
    ineq1_rhs = B * B
    ineq2_lhs = B * D
    ineq2_rhs = C * C
    res = {
        "CMV_P3": int(A),
        "CMV_P2Q": int(B),
        "CMV_PQ2": int(C),
        "CMV_Q3": int(D),
        "ineq1_holds": (ineq1_lhs <= ineq1_rhs),  # A*C <= B^2
        "ineq2_holds": (ineq2_lhs <= ineq2_rhs),  # B*D <= C^2
        "ineq1_lhs": int(ineq1_lhs),
        "ineq1_rhs": int(ineq1_rhs),
        "ineq2_lhs": int(ineq2_lhs),
        "ineq2_rhs": int(ineq2_rhs),
    }
    if verbose:
        print("== Degree-3 bivariate CMVs ==")
        print(f" CMV(P[3])   = {A}")
        print(f" CMV(P[2],Q) = {B}")
        print(f" CMV(P,Q[2]) = {C}")
        print(f" CMV(Q[3])   = {D}")
        print("\n== Inequalities you want to try to break ==")
        print(f"(1) A*C <= B^2  :  {ineq1_lhs} <= {ineq1_rhs}   -> {res['ineq1_holds']}")
        print(f"(2) B*D <= C^2  :  {ineq2_lhs} <= {ineq2_rhs}   -> {res['ineq2_holds']}")
    return res

# ----------------------------
# Helpers to build polytopes
# ----------------------------

def poly_from_vertices(vertices):
    """
    vertices: list of d-tuples of integers.
    """
    P = Polyhedron(vertices=vertices)
    if not is_bounded_poly(P):
        raise ValueError("Provided vertices do not define a bounded polytope.")
    return P

def random_lattice_polytope(d, m=10, box=2, seed=None, max_tries=300):
    """
    Quick helper to generate a random full-dim lattice polytope in Z^d.
    """
    import random
    rng = random.Random(seed)
    for _ in range(max_tries):
        pts = set()
        while len(pts) < m:
            pts.add(tuple(rng.randint(-box, box) for _ in range(d)))
        P = Polyhedron(vertices=list(pts))
        if is_bounded_poly(P) and P.dimension() == d:
            return P
    raise RuntimeError("Failed to generate a full-dimensional random lattice polytope.")

# ----------------------------
# Example usage (comment/uncomment as needed)
# ----------------------------
if __name__ == "__main__":
    # EXAMPLE: random 6D test (requires fast backend; no enumeration by default)
    d = 5
    P = Polyhedron(vertices=[(0, 0, 0, 0, 0), (1, 0, 0, 0, 0), (0, 1, 0, 0, 0), (0, 0, 1, 0, 0), (0, 0, 0, 1, 0), (1, 1, 1, 1, 5)])
    Q = Polyhedron(vertices=[(0, 0, 0, 0, 0), (1, 0, 0, 0, 0), (0, 1, 0, 0, 0), (0, 0, 1, 0, 0), (0, 0, 0, 1, 0), (1, -4, -1, 1, 1)])

    #P = random_lattice_polytope(d, m=7, box=2, seed=None)
    #Q = random_lattice_polytope(d, m=7, box=2, seed=None)

    # If you don't have barvinok/latte/normaliz set up and still want to try,
    # pass allow_enumeration_highdim=True (risky/slow for d>=4)
    out = check_degree3_inequalities(P, Q, allow_enumeration_highdim=True, verbose=True)
    print(out)


== Degree-3 bivariate CMVs ==
 CMV(P[3])   = 15
 CMV(P[2],Q) = 59
 CMV(P,Q[2]) = 59
 CMV(Q[3])   = 10

== Inequalities you want to try to break ==
(1) A*C <= B^2  :  885 <= 3481   -> True
(2) B*D <= C^2  :  590 <= 3481   -> True
{'CMV_P3': 15, 'CMV_P2Q': 59, 'CMV_PQ2': 59, 'CMV_Q3': 10, 'ineq1_holds': True, 'ineq2_holds': True, 'ineq1_lhs': 885, 'ineq1_rhs': 3481, 'ineq2_lhs': 590, 'ineq2_rhs': 3481}


In [11]:
P = Polyhedron(vertices=[(0, 0, 0, 0, 0), (1, 0, 0, 0, 0), (0, 1, 0, 0, 0), (0, 0, 1, 0, 0), (0, 0, 0, 1, 0), (1, 1, 1, 1, 5)])
(2*P).plot()

NotImplementedError: plotting of 5-dimensional polyhedra not implemented

In [31]:
from sage.interfaces.latte import count
def reeveEhr(q, k):
    return ((q+1)/6)*k**3 + k**2 + ((11-q)/6)*k + 1
p=8
q=7
P = Polyhedron(vertices = [(0, 0, 0), (1, 0, 0), (0, 1, 0), (1, q, q+1)])
Q = Polyhedron(vertices = [(0, 0, 0), (1, 0, 0), (0, 1, 0), (1, p, p+1)])
msum = P + Q
lhs = (msum.integral_points_count() - reeveEhr(q, 1) - reeveEhr(p,1) + 1)**2
rhs = (reeveEhr(q, 2) - 2*reeveEhr(q, 1) + 1)*(reeveEhr(p, 2) - 2*reeveEhr(p, 1) + 1)
print(lhs, rhs)


169 110


In [43]:
var('x,y,a,b')
t = 1 + (1/2)*a + (1/12)*(a^2 + b) + (1/24)*a*b
expand((x + (1/2)*x^2 + (1/6)*x^3)^2*t)

1/432*a^2*x^6 + 1/864*a*b*x^6 + 1/72*a^2*x^5 + 1/144*a*b*x^5 + 1/72*a*x^6 + 1/432*b*x^6 + 7/144*a^2*x^4 + 7/288*a*b*x^4 + 1/12*a*x^5 + 1/72*b*x^5 + 1/36*x^6 + 1/12*a^2*x^3 + 1/24*a*b*x^3 + 7/24*a*x^4 + 7/144*b*x^4 + 1/6*x^5 + 1/12*a^2*x^2 + 1/24*a*b*x^2 + 1/2*a*x^3 + 1/12*b*x^3 + 7/12*x^4 + 1/2*a*x^2 + 1/12*b*x^2 + x^3 + x^2

In [63]:
var('a,b,c,x,y,z')

(a, b, c, x, y, z)

In [66]:
R = PolynomialRing(QQ, 6, 'abcxyz')
lhs = ((a+x+1)*(b+y+1)*(c+z+1)-(a+1)*(b+1)*(c+1) - (x+1)*(y+1)*(z+1) + 1)^2

(a, b, c, x, y, z)

In [84]:
from sage.symbolic.relation import solve_ineq

In [111]:
var('x,y,t,a,b,c')
lhs = ((a+x+1)*(b+y+1)*(c+t+1) - (x+1)*(y+1)*(t+1) - (a+1)*(b+1)*(c+1) + 1)^2
rhs = (((2*a+1)*(2*b+1)*(2*c+1) - 2*(a+1)*(b+1)*(c+1) + 1) * ((2*x+1)*(2*y+1)*(2*t+1) - 2*(x+1)*(y+1)*(t+1) + 1))
assume(a>0,b>0,c>0,x>0,y>0,t>0)
val = rhs - lhs # want val > 0
eqn = (lhs < rhs)
solve_ineq([lhs < rhs], [a, b, c, x, y, t])

[[-a^2*b^2*t^2 - 2*a*b^2*c*t*x - 2*a*b^2*t^2*x - b^2*c^2*x^2 - 2*b^2*c*t*x^2 - b^2*t^2*x^2 - 2*a^2*b*c*t*y - 2*a^2*b*t^2*y - 2*a*b*c^2*x*y + 30*a*b*c*t*x*y - 2*a*b*t^2*x*y - 2*b*c^2*x^2*y - 2*b*c*t*x^2*y - a^2*c^2*y^2 - 2*a^2*c*t*y^2 - a^2*t^2*y^2 - 2*a*c^2*x*y^2 - 2*a*c*t*x*y^2 - c^2*x^2*y^2 - 2*a^2*b*t^2 - 2*a*b^2*t^2 - 2*a*b^2*t*x + 8*a*b*c*t*x - 2*b^2*c*t*x - 2*a*b*t^2*x - 2*b^2*t^2*x - 2*b^2*c*x^2 - 2*b*c^2*x^2 - 2*b^2*t*x^2 - 2*b*c*t*x^2 - 2*a^2*b*t*y - 2*a^2*c*t*y + 8*a*b*c*t*y - 2*a^2*t^2*y - 2*a*b*t^2*y + 8*a*b*c*x*y - 2*a*c^2*x*y - 2*b*c^2*x*y + 8*a*b*t*x*y + 8*a*c*t*x*y + 8*b*c*t*x*y - 2*b*c*x^2*y - 2*c^2*x^2*y - 2*a^2*c*y^2 - 2*a*c^2*y^2 - 2*a^2*t*y^2 - 2*a*c*t*y^2 - 2*a*c*x*y^2 - 2*c^2*x*y^2 - a^2*t^2 - 2*a*b*t^2 - b^2*t^2 + 2*a*b*t*x - 2*b^2*t*x + 2*a*c*t*x + 2*b*c*t*x - b^2*x^2 - 2*b*c*x^2 - c^2*x^2 - 2*a^2*t*y + 2*a*b*t*y + 2*a*c*t*y + 2*b*c*t*y + 2*a*b*x*y + 2*a*c*x*y + 2*b*c*x*y - 2*c^2*x*y - a^2*y^2 - 2*a*c*y^2 - c^2*y^2 > 0]]

In [115]:
for i in range(10):
    L = [RealField(10).random_element(0,100) for _ in range(6)]
    k = val({a:L[0],b:L[1],c:L[2],x:L[3],y:L[4],t:L[5]})
    print(k)
    if k > 0:
        print("hit:", L)
    
    

-6.4e11
-3.8e11
-6.4e9
-2.5e10
-6.8e10
-3.7e10
-2.6e11
-6.6e9
-3.6e12
-5.4e10


In [138]:
hess = val.hessian()

In [176]:
J = jacobian(val({x:1, b:1, c:0}), (y, t, a))

In [177]:
solve([J[0,0]==0,J[0,1]==0,J[0,2]==0], y, t, a)

[[y == (-1/2), t == (-1/2), a == 0],
 [y == -2, t == (-1/7), a == (-1/7)],
 [y == 1, t == 0, a == 1],
 [y == (-1.4183499514204838e+16),
  t == -3.107724425887265,
  a == (5.37286182890153e-16)],
 [y == 2096565219584893.0,
  t == 0.2615241635687732,
  a == (6.50618634375405e-16)],
 [y == -60345651283119.0,
  t == -0.15379978298436683,
  a == (-4.25107540496336e-15)]]

In [164]:
var('r')
hess({x:r, y:r, t:r, a:r,b:r, c:r}).eigenvalues()

[-12*r^4, -48*r^4 - 72*r^3 - 24*r^2, -48*r^4 - 72*r^3 - 24*r^2, 0, 0, 0]

In [175]:
round(3/22*sqrt(1505) - 35/22, ndigits=3)

3.699

In [184]:
val({x:1,y:20,t:0.2615241635687732,a:1, b:1, c:0})

-579.277983444120

In [181]:
hess({x:1,y:2096565219584893.0,t:0.2615241635687732,a:0, b:1, c:0}).eigenvalues()

ArithmeticError: could not determine eigenvalues exactly using symbolic matrices; try using a different type of matrix via self.change_ring(), if possible